# Step 3: Train TinyBERT Student Model
Run all cells in order. Works locally (CPU) or on Google Colab (GPU).

In [21]:
# Install dependencies (skip if already installed in venv)
!pip install transformers datasets torch scikit-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\sindh\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [22]:
import json
import torch
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from collections import Counter
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [23]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
INPUT_FILE    = "labeled_data.json"
MODEL_OUT_DIR = "crypto_student_model"
BASE_MODEL    = "huawei-noah/TinyBERT_General_4L_312D"  # ~56MB
LABEL_MAP     = {"BUY": 0, "SELL": 1, "HOLD": 2}
ID2LABEL      = {0: "BUY", 1: "SELL", 2: "HOLD"}
EPOCHS        = 5
BATCH_SIZE    = 16   # lowered for CPU; increase to 32 on GPU
LR            = 2e-5
MAX_LEN       = 128
DISTILL_TEMP  = 4.0
DISTILL_ALPHA = 0.7
# ──────────────────────────────────────────────────────────────────────────────

In [24]:
# ── LOAD & INSPECT DATA ───────────────────────────────────────────────────────
with open(INPUT_FILE) as f:
    data = json.load(f)

data = [s for s in data if s.get("label") in LABEL_MAP]
print(f"Total labeled samples: {len(data)}")
print(f"Label distribution: {dict(Counter(s['label'] for s in data))}")

Total labeled samples: 540
Label distribution: {'HOLD': 385, 'SELL': 75, 'BUY': 80}


In [25]:
# ── DATASET ───────────────────────────────────────────────────────────────────
def build_text(sample):
    fg = sample.get("fear_greed_index", 50)
    fg_label = (
        "extreme fear" if fg < 25 else
        "fear"         if fg < 45 else
        "neutral"      if fg < 55 else
        "greed"        if fg < 75 else
        "extreme greed"
    )
    return (
        f"{sample['coin'].upper()} price ${sample['price']:,}, "
        f"24h change {sample['price_change_24h_pct']}%, "
        f"fear greed {fg} ({fg_label}). "
        f"News: {sample['headline']}"
    )

class CryptoDataset(Dataset):
    def __init__(self, samples, tokenizer, max_len=MAX_LEN):
        self.samples   = samples
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s   = self.samples[idx]
        enc = self.tokenizer(
            build_text(s),
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "label":          torch.tensor(LABEL_MAP[s["label"]], dtype=torch.long),
        }

In [26]:
# ── DISTILLATION LOSS ─────────────────────────────────────────────────────────
def distillation_loss(student_logits, true_labels, temperature=DISTILL_TEMP, alpha=DISTILL_ALPHA):
    hard_loss = F.cross_entropy(student_logits, true_labels)
    num_classes = student_logits.size(-1)
    smooth_labels = torch.full_like(student_logits, fill_value=0.1 / (num_classes - 1))
    smooth_labels.scatter_(1, true_labels.unsqueeze(1), 0.9)
    soft_student = F.log_softmax(student_logits / temperature, dim=-1)
    soft_loss    = F.kl_div(soft_student, smooth_labels, reduction="batchmean")
    return alpha * soft_loss + (1 - alpha) * hard_loss

In [27]:
# ── LOAD MODEL ────────────────────────────────────────────────────────────────
print(f"Loading: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model     = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=3, id2label=ID2LABEL, label2id=LABEL_MAP,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,} ({total_params/1e6:.1f}M)")

Loading: huawei-noah/TinyBERT_General_4L_312D


Loading weights: 100%|██████████| 71/71 [00:00<00:00, 11838.43it/s]
BertForSequenceClassification LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     | 
-------------------------------------------+------------+-
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	

Parameters: 14,351,187 (14.4M)


In [28]:
# ── TRAIN / VAL SPLIT ─────────────────────────────────────────────────────────
train_data, val_data = train_test_split(data, test_size=0.15, random_state=42)
print(f"Train: {len(train_data)}  Val: {len(val_data)}")
print(f"Train distribution: {dict(Counter(s['label'] for s in train_data))}")

train_ds = CryptoDataset(train_data, tokenizer)
val_ds   = CryptoDataset(val_data,   tokenizer)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

Train: 459  Val: 81
Train distribution: {'HOLD': 327, 'SELL': 66, 'BUY': 66}


In [29]:
# ── TRAINING LOOP ─────────────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_dl) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps,
)

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids    = batch["input_ids"].to(device)
            mask   = batch["attention_mask"].to(device)
            logits = model(ids, attention_mask=mask).logits
            all_preds.extend(logits.argmax(-1).cpu().tolist())
            all_labels.extend(batch["label"].tolist())
    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    return acc, all_preds, all_labels

best_val_acc = 0.0
print(f"Training for {EPOCHS} epochs on {device}...\n")

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for step, batch in enumerate(train_dl):
        ids    = batch["input_ids"].to(device)
        mask   = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model(ids, attention_mask=mask).logits
        loss   = distillation_loss(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    val_acc, preds, labels_list = evaluate(model, val_dl)
    avg_loss = total_loss / len(train_dl)
    print(f"Epoch {epoch}/{EPOCHS} — Loss: {avg_loss:.4f}  Val Acc: {val_acc:.4f}")
    print(classification_report(labels_list, preds, target_names=["BUY", "SELL", "HOLD"], zero_division=0))

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained(MODEL_OUT_DIR)
        tokenizer.save_pretrained(MODEL_OUT_DIR)
        print(f"  ★ Best model saved to {MODEL_OUT_DIR}/\n")

print(f"\n✓ Done! Best val accuracy: {best_val_acc:.4f}")
print(f"  Model saved to: {MODEL_OUT_DIR}/")
print("  Next: run step4_export_model.py")

Training for 5 epochs on cpu...

Epoch 1/5 — Loss: 0.7967  Val Acc: 0.7160
              precision    recall  f1-score   support

         BUY       0.00      0.00      0.00        14
        SELL       0.00      0.00      0.00         9
        HOLD       0.72      1.00      0.83        58

    accuracy                           0.72        81
   macro avg       0.24      0.33      0.28        81
weighted avg       0.51      0.72      0.60        81



Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 10.89it/s]

  ★ Best model saved to crypto_student_model/



Epoch 2/5 — Loss: 0.7253  Val Acc: 0.7160
              precision    recall  f1-score   support

         BUY       0.00      0.00      0.00        14
        SELL       0.00      0.00      0.00         9
        HOLD       0.72      1.00      0.83        58

    accuracy                           0.72        81
   macro avg       0.24      0.33      0.28        81
weighted avg       0.51      0.72      0.60        81

Epoch 3/5 — Loss: 0.6859  Val Acc: 0.7160
              precision    recall  f1-score   support

         BUY       0.00      0.00      0.00        14
        SELL       0.00      0.00      0.00         9
        HOLD       0.72      1.00      0.83        58

    accuracy                           0.72        81
   macro avg       0.24      0.33      0.28        81
weighted avg       0.51      0.72      0.60        81

Epoch 4/5 — Loss: 0.6710  Val Acc: 0.7160
              precision    recall  f1-score   support

         BUY       0.00      0.00      0.00        14
   